# Mann–Whitney U Test for Two Independent Samples

**STA 507 1.0 Nonparametric Statistics** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta507.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Design and question](#theory-01)
- [Pooling and ranking](#theory-02)
- [Hypotheses and tails](#theory-03)
- [Exact and large-sample references](#theory-04)
- [Estimation and interpretation](#theory-05)
- [Reporting and cautions](#theory-06)

## Detailed theory · Mann–Whitney U for independent groups

<a id="theory-01"></a>

### Design and question
Mann–Whitney uses two independent samples with at least an ordered outcome. The standard null is equality of their distributions; an ordered alternative asks whether one tends to produce larger observations. Paired data require a paired method. The test is often described as comparing medians, but that interpretation needs a suitable common-shape location-shift model.

<a id="theory-02"></a>

### Pooling and ranking
Pool both groups, rank from smallest to largest, and average tied ranks. Keep group labels. If group A has size m and rank sum $R_A$:
$$U_A=R_A-\frac{m(m+1)}2,\quad
U_B=mn-U_A.$$
The rank sum check is $R_A+R_B=(m+n)(m+n+1)/2$. Under this convention $U_A$ counts A>B pairs, with half-credit for ties. Large $U_A$ favours larger A values. Some books use the complementary U, so reconcile definitions before using a table.

<a id="theory-03"></a>

### Hypotheses and tails
A two-sided test detects either distributional ordering direction. An A-greater alternative uses the upper tail of the displayed $U_A$; A-less uses its lower tail. A minimum-U critical table is typically two-sided and loses direction, just as the minimum signed-rank sum does.

<a id="theory-04"></a>

### Exact and large-sample references
With no ties, under exchangeable group labels, enumerate the $\binom{m+n}{m}$ rank allocations or use an exact U algorithm. For tied values a no-ties exact table is not automatically valid; a tie-aware permutation reference retains the observed pooled values.
$$E_0(U_A)=mn/2,\quad Var_0(U_A)=mn(m+n+1)/12\quad\text{without ties}.$$
If pooled tie groups have sizes $t_j$ and $N=m+n$:
$$Var_0(U_A)=\frac{mn}{12}\left[N+1-\frac{\sum_j(t_j^3-t_j)}{N(N-1)}\right].$$
Declare any normal-approximation continuity correction and check whether it is suitable for the available sample sizes and ties.

<a id="theory-05"></a>

### Estimation and interpretation
$U_A/(mn)$ estimates the probability that an A observation exceeds a B observation, counting ties by half, under the displayed convention. A rank-biserial form is $2U_A/(mn)-1$. These are ordinal summaries, not mean differences.

<a id="theory-06"></a>

### Reporting and cautions
Give group sizes, descriptive distributions, rank sums, U convention, alternative, p-value method and context. Similar medians with different spreads may still yield a distributional difference. A nonsignificant test does not establish identical distributions or equal medians.

## Concepts covered

**Source alignment:** Handout 1: Mann–Whitney U test for two independent samples (week 4).

**Mission:** Match each design and measurement scale to a test, then explain the result in context.

## Worked calculation · Mann–Whitney U: pooled ranks and group rank sums

### Question and assumptions

Independent groups A=[1,3,5] and B=[2,4,6] have no ties. Test equality of distributions with a two-sided exact permutation test.

### Procedure

1. Pool the values, retain group labels, and rank all six values together.
2. Add ranks within each group.
3. Use U_A=R_A−n_A(n_A+1)/2 and U_B=n_An_B−U_A.
4. Enumerate all ways to allocate three of the six fixed ranks to A. Compare equally extreme rank sums.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Pooled value | Group | Rank |
| --- | --- | --- |
| 1 | A | 1 |
| 2 | B | 2 |
| 3 | A | 3 |
| 4 | B | 4 |
| 5 | A | 5 |
| 6 | B | 6 |

| Group | n | Rank sum | U |
| --- | --- | --- | --- |
| A | 3 | 9 | 3 |
| B | 3 | 12 | 6 |

| Quantity | Value |
| --- | --- |
| Allocations | 20 |
| Exact two-sided p | 0.7 |
| Decision at .05 | Do not reject H0 |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


A=[1,3,5]; B=[2,4,6]
pooled=sorted([(x,"A") for x in A]+[(x,"B") for x in B])
ranks=average_ranks([x for x,g in pooled])
show_table(["Pooled value","Group","Rank"],[[x,g,r] for (x,g),r in zip(pooled,ranks)])
RA=sum(r for (x,g),r in zip(pooled,ranks) if g=="A"); RB=sum(ranks)-RA
nA=len(A); nB=len(B); UA=RA-nA*(nA+1)/2; UB=nA*nB-UA
null_U=[sum(c)-nA*(nA+1)/2 for c in combinations(ranks,nA)]
center=nA*nB/2; p=sum(abs(u-center)>=abs(UA-center)-1e-12 for u in null_U)/len(null_U)
show_table(["Group","n","Rank sum","U"],[["A",nA,RA,UA],["B",nB,RB,UB]])
show_table(["Quantity","Value"],[["Allocations",len(null_U)],["Exact two-sided p",p],["Decision at .05","Reject H0" if p<=.05 else "Do not reject H0"]])
assert RA+RB==21 and UA+UB==9 and len(null_U)==20


### Decision and interpretation

Define which group U refers to: SciPy returns U for the first input. A location or median interpretation needs additional comparable-shape assumptions. With ties, average ranks and use an appropriate tie-aware permutation or asymptotic procedure; the standard no-ties exact U distribution does not correct ties. [SciPy Mann–Whitney conventions](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mannwhitneyu.html).

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(507)

In [ ]:
a=rng.lognormal(2,.35,25); b=rng.lognormal(2.2,.35,27)
res=stats.mannwhitneyu(a,b,alternative='two-sided',method='auto')
print('U statistic:',res.statistic,'p-value:',res.pvalue)
print('medians:',np.median(a).round(2),np.median(b).round(2))
plt.boxplot([a,b],tick_labels=['group A','group B']); plt.title('Show distributions with the rank test'); plt.show()

### Practice · Rank arena
1. Two independent samples have n₁=8, n₂=10 and rank sum R₁=75. Calculate U₁ and U₂.
2. State the identical-distribution null interpretation.
3. Why is “Mann–Whitney tests equality of medians” an incomplete statement?

<details><summary>Answer sketches</summary>
1. U₁=75−8·9/2=39; U₂=80−39=41. 2. Under H₀, group labels are exchangeable across the pooled ranks. 3. It is generally a distributional/rank comparison; medians are the focus only with comparable shapes and a location-shift interpretation.
</details>